# Experiment 5.0.1 — Analog-Head Control

Analysis-only notebook. Training is performed by a 6-task Slurm array. This notebook aggregates finalized results for two analog-head controls and compares them with historical Exp3.0.5 and Exp5.0 references.

The two new controls separate hidden-dynamics implementation from the spiking-output path: `exp3_synaptic_analog` and `exp5_macro_binary_analog`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'notebooks').is_dir():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_5_0_1_exp3_analog_head_control' / 'analog_head_hidden_dynamics_control_v2'
runs = pd.read_csv(ART / 'runs.csv')
comparison = pd.read_csv(ART / 'comparison_runs.csv')
manifest = json.loads((ART / 'manifest.json').read_text(encoding='utf-8'))
display(manifest)
display(runs)

## New-control aggregate

Both conditions use seeds `(11, 23, 101)`. Aggregation is performed here in the notebook from finalized per-seed rows.

In [ ]:
primary_columns = [
    'native_test_ba',
    'full_count_test_ba',
    'fixed250_ordered_test_ba',
    'relative10_ordered_test_ba',
]
new_summary = runs.groupby('condition')[primary_columns].agg(['mean', 'std', 'count'])
display(new_summary)

## Unified comparison

`native_or_output_ba` is native analog-head segment BA for Exp3/new controls and Output WholeCount BA for the historical Exp5 spiking-output models. The three frozen L2 probe metrics are directly comparable representation diagnostics.

In [ ]:
metric_columns = [
    'native_or_output_ba',
    'full_count_ba',
    'fixed250_ordered_ba',
    'relative10_ordered_ba',
]
comparison_summary = (
    comparison.groupby(['source', 'variant'])[metric_columns]
    .agg(['mean', 'std', 'count'])
)
display(comparison_summary)

## Gate 1 — exact Exp3 reproduction

Compare `exp3_synaptic_analog` against historical Exp3.0.5 on all three paired seeds. Near-zero deltas indicate that the historical Exp3 training/probe pipeline has been reproduced.

In [ ]:
new_exp3 = comparison[(comparison['source'] == 'exp5_0_1_control') & (comparison['variant'] == 'exp3_synaptic_analog')].set_index('seed')
historical_exp3 = comparison[comparison['source'] == 'exp3_0_5_reference'].set_index('seed')
common = sorted(set(new_exp3.index) & set(historical_exp3.index))
paired_reproduction = pd.DataFrame(index=common)
for metric in metric_columns:
    paired_reproduction[metric] = new_exp3.loc[common, metric] - historical_exp3.loc[common, metric]
display(paired_reproduction)
display(pd.DataFrame({'mean_delta': paired_reproduction.mean(), 'sd_delta': paired_reproduction.std()}))

## Gate 2 — hidden implementation effect

Both new models use the same analog timestep head and exact same labels/split. Their paired difference isolates the effect of using Exp3 `snnTorch.Synaptic` hidden dynamics versus Exp5 Macro binary hidden dynamics more cleanly than comparing either against a spiking output model.

In [ ]:
new_macro = comparison[(comparison['source'] == 'exp5_0_1_control') & (comparison['variant'] == 'exp5_macro_binary_analog')].set_index('seed')
common_hidden = sorted(set(new_exp3.index) & set(new_macro.index))
paired_hidden = pd.DataFrame(index=common_hidden)
for metric in metric_columns:
    paired_hidden[metric] = new_macro.loc[common_hidden, metric] - new_exp3.loc[common_hidden, metric]
display(paired_hidden)
display(pd.DataFrame({'mean_delta_macro_minus_exp3': paired_hidden.mean(), 'sd_delta': paired_hidden.std()}))

## Gate 3 — analog head versus Exp5 spiking-output path

The Exp5.0 reference uses seeds `(11,23,37,53,71)`, while the new controls use `(11,23,101)`. Only seeds `11` and `23` are treated as paired. Compare the new `exp5_macro_binary_analog` representation with existing Exp5 `timestep_ce + binary`; Multi-HO is shown as an output-capacity reference.

In [ ]:
paired_exp5_tables = {}
for variant in ('binary', 'multi_ho'):
    ref = comparison[(comparison['source'] == 'exp5_0_spiking_output_reference') & (comparison['variant'] == variant)].set_index('seed')
    common_exp5 = sorted(set(new_macro.index) & set(ref.index))
    delta = pd.DataFrame(index=common_exp5)
    for metric in metric_columns:
        delta[metric] = new_macro.loc[common_exp5, metric] - ref.loc[common_exp5, metric]
    paired_exp5_tables[variant] = delta
    print(f'Exp5-Macro analog - Exp5 {variant}, paired seeds={common_exp5}')
    display(delta)
    display(pd.DataFrame({'mean_delta': delta.mean(), 'sd_delta': delta.std()}))

## Representation comparison plot

In [ ]:
plot_metrics = ['full_count_ba', 'fixed250_ordered_ba', 'relative10_ordered_ba']
labels = {
    'full_count_ba': 'L2 FullCount + Linear',
    'fixed250_ordered_ba': 'L2 Fixed250 + Linear',
    'relative10_ordered_ba': 'L2 Relative10 + Linear',
}
plot_frame = comparison.groupby(['source', 'variant'])[plot_metrics].mean().reset_index()
plot_frame['model'] = plot_frame['source'] + ' / ' + plot_frame['variant']
x = np.arange(len(plot_metrics))
width = 0.8 / len(plot_frame)
fig, ax = plt.subplots(figsize=(13, 6))
for i, row in plot_frame.iterrows():
    values = [100 * row[m] for m in plot_metrics]
    ax.bar(x + (i - (len(plot_frame)-1)/2) * width, values, width=width, label=row['model'])
ax.set_xticks(x, [labels[m] for m in plot_metrics])
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_title('Exp5.0.1 analog-head controls vs Exp3 / Exp5 references')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Interpretation logic

1. If `exp3_synaptic_analog` does not reproduce historical Exp3, stop and diagnose protocol drift.
2. If reproduction succeeds and `exp5_macro_binary_analog` stays close to it, hidden implementation is not the main cause.
3. If the Macro analog model is strong but existing Exp5 binary is weak, the timestep-supervision degradation is localized to the spiking output path / its checkpoint-selection regime.
4. If Macro analog itself is much weaker than Exp3 analog, hidden dynamics/surrogate implementation contributes materially and the output LIF cannot be blamed alone.